# Day 9 — Supervised vs unsupervised learning: regression vs classification

Run each cell with **Shift + Enter**, from the top. We use the 150-student dataset from Days 5-8, `student_data_150.csv`: 8 real students and 142 **synthetic** ones (simulated for practice, marked in the Source column).

## 1. Supervised vs unsupervised learning

In [ ]:
# Get the dataset: Colab starts with an empty folder, so download the file the first time
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-009/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import pandas as pd

# The same 150 students as Days 5-8: 8 real ones and 142 synthetic ones
df = pd.read_csv("student_data_150.csv")

# Supervised learning: features (inputs) and a known target (the answer)
X = df[["StudyHours", "Attendance", "Age"]]
y = df["ExamScore"]
print("Features:")
print(X.head())
print("\nKnown target:")
print(y.head().to_string())

Features:
   StudyHours  Attendance   Age
0         1.0        60.0  10.0
1         2.0        70.0  11.0
2         3.0        80.0  11.0
3         4.0        80.0  12.0
4         3.0        95.0  11.0

Known target:
0    35.0
1    45.0
2    55.0
3    65.0
4    75.0


In [ ]:
# Regression or classification? It depends on the target
print("ExamScore: a number with", df["ExamScore"].nunique(), "different values, from",
      df["ExamScore"].min(), "to", df["ExamScore"].max(), "-> regression")
print("Passed: a category with", df["Passed"].nunique(), "values", sorted(df["Passed"].unique().tolist()), "-> classification")
print(df["Passed"].value_counts().to_string())

ExamScore: a number with 51 different values, from 24.0 to 91.0 -> regression
Passed: a category with 2 values [False, True] -> classification
Passed
True     106
False     44


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Unsupervised learning: no target, just look for groups
features = df[["StudyHours", "Attendance", "Age"]]
X_scaled = StandardScaler().fit_transform(features)      # put the columns on the same scale
model = KMeans(n_clusters=3, random_state=42, n_init=10)
df["Cluster"] = model.fit_predict(X_scaled)
print(df[["Student", "StudyHours", "Attendance", "Age", "Cluster"]].head().to_string(index=False))
print("\nStudents per cluster:", df["Cluster"].value_counts().sort_index().to_dict())

Student  StudyHours  Attendance  Age  Cluster
     S1         1.0        60.0 10.0        1
     S2         2.0        70.0 11.0        1
     S3         3.0        80.0 11.0        1
     S4         4.0        80.0 12.0        2
     S5         3.0        95.0 11.0        0

Students per cluster: {0: 49, 1: 48, 2: 53}


In [ ]:
# What did the clusters find? Look at their characteristics before naming them
profile = df.groupby("Cluster")[["StudyHours", "Attendance", "Age", "ExamScore"]].mean().round(1)
profile["Students"] = df["Cluster"].value_counts().sort_index()
print(profile)
df[["Student", "Cluster", "StudyHours", "Attendance", "Age", "ExamScore"]].assign(
    Cluster=lambda d: "Cluster " + d["Cluster"].astype(str)).to_csv("student_clusters.csv", index=False)

         StudyHours  Attendance   Age  ExamScore  Students
Cluster                                                   
0               4.6        88.4  10.7       66.2        49
1               2.5        73.8  10.5       49.0        48
2               3.7        76.3  11.8       56.8        53


## 2. Real-world problem examples

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# One 70/15/15 split for the examples, stratified on Passed so both classes appear in every set
train_val, test = train_test_split(df, test_size=0.15, random_state=42, stratify=df["Passed"])
train, validation = train_test_split(train_val, test_size=0.15 / 0.85, random_state=42,
                                     stratify=train_val["Passed"])

# Example A, regression: predict the exam score, a number
features = ["StudyHours", "Attendance", "Age"]
model = LinearRegression().fit(train[features], train["ExamScore"])
prediction = model.predict(validation[features])
print("Predicted:", prediction[:5].round(1))
print("Actual:   ", validation["ExamScore"].head().to_numpy())

Predicted: [57.4 55.  55.4 52.4 51.9]
Actual:    [61. 41. 36. 35. 52.]


In [ ]:
from sklearn.linear_model import LogisticRegression

# Example B, classification: predict Passed (ExamScore of 50 or more, this dataset's rule)
classifier = LogisticRegression().fit(train[features], train["Passed"])
predictions = classifier.predict(validation[features])
print("Predicted:", predictions[:5])
print("Actual:   ", validation["Passed"].head().to_numpy())
accuracy = (predictions == validation["Passed"]).mean()
baseline = validation["Passed"].mean()                  # always predicting "passed"
print(f"Validation accuracy: {accuracy:.2f} | always predicting pass: {baseline:.2f}")

Predicted: [ True  True  True  True  True]
Actual:    [ True False False False  True]
Validation accuracy: 0.61 | always predicting pass: 0.70


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Example C, unsupervised: group students without any target
X_scaled = StandardScaler().fit_transform(df[features])
groups = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)
print(df.assign(Cluster=groups).groupby("Cluster")[features].mean().round(1))

         StudyHours  Attendance   Age
Cluster                              
0               4.6        88.4  10.7
1               2.5        73.8  10.5
2               3.7        76.3  11.8


In [ ]:
# The production note, demonstrated: Passed is made FROM ExamScore, so ExamScore must not be a feature
leaky = LogisticRegression(max_iter=1000).fit(train[features + ["ExamScore"]], train["Passed"])
leaky_accuracy = (leaky.predict(validation[features + ["ExamScore"]]) == validation["Passed"]).mean()
print(f"With ExamScore as a feature: validation accuracy {leaky_accuracy:.2f} (the answer leaked in)")
print(f"Without it:                  validation accuracy {accuracy:.2f}")

With ExamScore as a feature: validation accuracy 1.00 (the answer leaked in)
Without it:                  validation accuracy 0.61


## 3. Data and target variables

In [ ]:
# Prepare X and y: one feature matrix, two possible targets
X = df[["StudyHours", "Attendance", "Age"]]
y_regression = df["ExamScore"]

df["Pass"] = (df["ExamScore"] >= 40).astype(int)     # an educational passing mark of 40
y_classification = df["Pass"]

print("Feature shape:", X.shape)
print("Regression target:", y_regression.head().tolist())
print("Classification target:", y_classification.head().tolist())

Feature shape: (150, 3)
Regression target: [35.0, 45.0, 55.0, 65.0, 75.0]
Classification target: [0, 1, 1, 1, 1]


In [ ]:
# The target is a choice: the dataset's Passed column uses 50, this exercise uses 40
print("Pass rate with a mark of 40:", round(df["Pass"].mean(), 3))
print("Pass rate with a mark of 50:", round(df["Passed"].mean(), 3))
changed = df[(df["Pass"] == 1) & (~df["Passed"])]
print("Students who pass at 40 but not at 50:", len(changed))

Pass rate with a mark of 40: 0.907
Pass rate with a mark of 50: 0.707
Students who pass at 40 but not at 50: 30


In [ ]:
# Which columns can be features? Check each one's role before modelling
roles = {
    "Student": "ID, not a feature (unique per student)",
    "Class": "candidate feature (category, needs encoding)",
    "Age": "candidate feature",
    "StudyHours": "candidate feature",
    "Attendance": "candidate feature",
    "PreviousScore": "candidate feature (known before the exam)",
    "ExamScore": "target for regression",
    "Passed": "derived from ExamScore: leaks the answer",
    "Source": "dataset label (real or synthetic), not a feature",
}
for col in df.columns.drop(["Pass", "Cluster"], errors="ignore"):
    print(f"{col:<14} {df[col].dtype!s:<8} {roles[col]}")

Student        object   ID, not a feature (unique per student)
Class          object   candidate feature (category, needs encoding)
Age            float64  candidate feature
StudyHours     float64  candidate feature
Attendance     float64  candidate feature
PreviousScore  int64    candidate feature (known before the exam)
ExamScore      float64  target for regression
Passed         bool     derived from ExamScore: leaks the answer
Source         object   dataset label (real or synthetic), not a feature


In [ ]:
from sklearn.metrics import r2_score

# Does an extra feature help? Compare on the validation set from Section 2
for cols in (["StudyHours", "Attendance", "Age"], ["StudyHours", "Attendance", "Age", "PreviousScore"]):
    m = LinearRegression().fit(train[cols], train["ExamScore"])
    print(f"{' + '.join(cols):<50} validation R2 {r2_score(validation['ExamScore'], m.predict(validation[cols])):.3f}")

StudyHours + Attendance + Age                      validation R2 0.274
StudyHours + Attendance + Age + PreviousScore      validation R2 0.401


## 4. Evaluation overview

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Regression metrics by hand: three students
y_true = [80, 60, 90]
y_pred = [75, 65, 85]
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
print("MAE:", mae, "| MSE:", mse, "| RMSE:", mse ** 0.5)
print("R2:", round(r2_score(y_true, y_pred), 3), "(errors of 5 are small next to how much these scores vary)")

MAE: 5.0 | MSE: 25.0 | RMSE: 5.0
R2: 0.839 (errors of 5 are small next to how much these scores vary)


In [ ]:
# Regression metrics for Section 2's model on the validation set
y_true, y_pred = validation["ExamScore"], model.predict(validation[features])
mse = mean_squared_error(y_true, y_pred)
print(f"MAE {mean_absolute_error(y_true, y_pred):.2f} | MSE {mse:.2f} | RMSE {mse ** 0.5:.2f} | R2 {r2_score(y_true, y_pred):.3f}")

MAE 9.01 | MSE 125.99 | RMSE 11.22 | R2 0.274


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Classification metrics for Section 2's classifier. Say which class is "positive" first.
y_true, y_pred = validation["Passed"], classifier.predict(validation[features])
print("Confusion matrix (rows: actual Fail, Pass; columns: predicted Fail, Pass):")
print(confusion_matrix(y_true, y_pred, labels=[False, True]))
for positive in [True, False]:
    name = "Passed" if positive else "Failed"
    print(f"Positive class = {name:<6} | precision {precision_score(y_true, y_pred, pos_label=positive, zero_division=0):.2f}"
          f" | recall {recall_score(y_true, y_pred, pos_label=positive):.2f} | F1 {f1_score(y_true, y_pred, pos_label=positive, zero_division=0):.2f}")
print(f"Accuracy {accuracy_score(y_true, y_pred):.2f}")

Confusion matrix (rows: actual Fail, Pass; columns: predicted Fail, Pass):
[[ 0  7]
 [ 2 14]]


Positive class = Passed | precision 0.67 | recall 0.88 | F1 0.76
Positive class = Failed | precision 0.00 | recall 0.00 | F1 0.00
Accuracy 0.61


In [ ]:
# The formulas, on the illustrative matrix: TP 84, FN 6, FP 8, TN 54 (positive = Pass)
TP, FN, FP, TN = 84, 6, 8, 54
precision, recall = TP / (TP + FP), TP / (TP + FN)
print(f"Accuracy  = (TP + TN) / all = {(TP + TN) / (TP + FN + FP + TN):.3f}")
print(f"Precision = TP / (TP + FP)  = {precision:.3f}")
print(f"Recall    = TP / (TP + FN)  = {recall:.3f}")
print(f"F1        = 2PR / (P + R)   = {2 * precision * recall / (precision + recall):.3f}")

Accuracy  = (TP + TN) / all = 0.908


Precision = TP / (TP + FP)  = 0.913
Recall    = TP / (TP + FN)  = 0.933
F1        = 2PR / (P + R)   = 0.923


In [ ]:
# Why accuracy can mislead: a "model" that always predicts pass
always_pass = [True] * len(validation)
print(f"Accuracy {accuracy_score(validation['Passed'], always_pass):.2f}"
      f" | recall for Failed {recall_score(validation['Passed'], always_pass, pos_label=False):.2f}"
      " (it never finds a student at risk)")

Accuracy 0.70 | recall for Failed 0.00 (it never finds a student at risk)


## 5. Choosing the right approach

In [ ]:
# The decision framework as code: no target -> unsupervised; numeric -> regression; categories -> classification
def choose_approach(target):
    if target is None:
        return "unsupervised (no target)"
    if pd.api.types.is_bool_dtype(target) or not pd.api.types.is_numeric_dtype(target):
        return f"classification ({target.nunique()} classes)"
    if target.nunique() <= 10:
        return f"check: numeric but only {target.nunique()} values (regression or classification?)"
    return f"regression ({target.nunique()} different values)"

for name in ["ExamScore", "Passed", "Class", "Age"]:
    print(f"Target {name:<10} -> {choose_approach(df[name])}")
print(f"No target         -> {choose_approach(None)}")

Target ExamScore  -> regression (51 different values)
Target Passed     -> classification (2 classes)
Target Class      -> classification (4 classes)
Target Age        -> check: numeric but only 3 values (regression or classification?)
No target         -> unsupervised (no target)


In [ ]:
from sklearn.metrics import silhouette_score

# Unsupervised learning needs evaluation too: the silhouette score (-1 to 1, higher = clearer groups)
X_habits = StandardScaler().fit_transform(df[["StudyHours", "Attendance"]])
for k in [2, 3, 4, 5]:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_habits)
    print(f"{k} clusters: silhouette {silhouette_score(X_habits, labels):.3f}")

2 clusters: silhouette 0.352
3 clusters: silhouette 0.343
4 clusters: silhouette 0.356
5 clusters: silhouette 0.350


## 6. Mini project: student performance analysis

In [ ]:
# Mini project: load and explore (8 real students, 142 synthetic)
df = pd.read_csv("student_data_150.csv")
features = ["StudyHours", "Attendance", "Age"]
df["Pass"] = (df["ExamScore"] >= 40).astype(int)          # the project's educational passing mark of 40
print("Shape:", df.shape, "| missing values:", int(df.isnull().sum().sum()))
print(df[features + ["ExamScore"]].describe().loc[["mean", "min", "max"]].round(1))
print("Pass rate:", round(df["Pass"].mean(), 3))

Shape: (150, 10) | missing values: 0
      StudyHours  Attendance   Age  ExamScore
mean         3.6        79.5  11.0       57.4
min          0.5        54.0  10.0       24.0
max          7.0       100.0  12.0       91.0
Pass rate: 0.907


In [ ]:
from sklearn.pipeline import make_pipeline

# A leakage-safe 70/15/15 split, stratified on Pass so every set has some failing students
train_val, test = train_test_split(df, test_size=0.15, random_state=42, stratify=df["Pass"])
train, validation = train_test_split(train_val, test_size=0.15 / 0.85, random_state=42, stratify=train_val["Pass"])
print("Train", len(train), "| validation", len(validation), "| test", len(test),
      "| failing students in validation:", int((validation["Pass"] == 0).sum()), "and test:", int((test["Pass"] == 0).sum()))

# Regression: compare with the mean baseline on validation, then test the chosen model once
from sklearn.dummy import DummyRegressor
regressors = {"Mean baseline": DummyRegressor(), "Linear regression": make_pipeline(StandardScaler(), LinearRegression())}
for name, m in regressors.items():
    m.fit(train[features], train["ExamScore"])
    p = m.predict(validation[features])
    print(f"{name:<18} validation MAE {mean_absolute_error(validation['ExamScore'], p):.2f}"
          f" RMSE {mean_squared_error(validation['ExamScore'], p) ** 0.5:.2f} R2 {r2_score(validation['ExamScore'], p):.3f}")
reg_model = regressors["Linear regression"]
p = reg_model.predict(test[features])
print(f"Chosen: linear regression | test MAE {mean_absolute_error(test['ExamScore'], p):.2f}"
      f" RMSE {mean_squared_error(test['ExamScore'], p) ** 0.5:.2f} R2 {r2_score(test['ExamScore'], p):.3f}")

Train 104 | validation 23 | test 23 | failing students in validation: 2 and test: 2
Mean baseline      validation MAE 10.34 RMSE 13.67 R2 -0.000
Linear regression  validation MAE 7.72 RMSE 10.35 R2 0.427
Chosen: linear regression | test MAE 8.11 RMSE 10.31 R2 0.516


In [ ]:
# Classification: 91% pass, so judge models by how well they find FAILING students (positive class = fail)
classifiers = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression()),
    "Logistic, balanced": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")),
}
def fail_metrics(y, pred):
    return (accuracy_score(y, pred), precision_score(y, pred, pos_label=0, zero_division=0),
            recall_score(y, pred, pos_label=0), f1_score(y, pred, pos_label=0, zero_division=0))
scores = {}
for name, m in classifiers.items():
    m.fit(train[features], train["Pass"])
    scores[name] = fail_metrics(validation["Pass"], m.predict(validation[features]))
    a, pr, rc, f1 = scores[name]
    print(f"{name:<20} validation accuracy {a:.2f} | fail precision {pr:.2f} recall {rc:.2f} F1 {f1:.2f}")
best = max(scores, key=lambda n: (scores[n][2], scores[n][3]))   # rule: most failing students found, then F1
a, pr, rc, f1 = fail_metrics(test["Pass"], classifiers[best].predict(test[features]))
print(f"Chosen on validation: {best} | test accuracy {a:.2f} | fail precision {pr:.2f} recall {rc:.2f} F1 {f1:.2f}")

Logistic regression  validation accuracy 0.96 | fail precision 1.00 recall 0.50 F1 0.67
Logistic, balanced   validation accuracy 0.78 | fail precision 0.29 recall 1.00 F1 0.44
Chosen on validation: Logistic, balanced | test accuracy 0.83 | fail precision 0.25 recall 0.50 F1 0.33


In [ ]:
from sklearn.metrics import silhouette_score

# Clustering: fit the scaler and K-Means on the training students, then assign new students
scaler = StandardScaler().fit(train[features])
X_train, X_test = scaler.transform(train[features]), scaler.transform(test[features])
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_train)
    print(f"{k} clusters: training silhouette {silhouette_score(X_train, km.labels_):.3f}")
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_train)
profile = train.assign(Cluster=kmeans.labels_).groupby("Cluster")[features + ["ExamScore"]].mean().round(1)
print(profile)
print("Test students per cluster:", pd.Series(kmeans.predict(X_test)).value_counts().sort_index().to_dict())

2 clusters: training silhouette 0.281


3 clusters: training silhouette 0.282


4 clusters: training silhouette 0.283


         StudyHours  Attendance   Age  ExamScore
Cluster                                         
0               3.6        73.7  11.7       57.1
1               4.4        88.5  10.8       64.5
2               2.3        73.7  10.2       47.4
Test students per cluster: {0: 9, 1: 11, 2: 3}


In [ ]:
# Save results for the charts: regression test predictions and training cluster sizes
pd.DataFrame({"ActualScore": test["ExamScore"], "PredictedScore": reg_model.predict(test[features]).round(1)}
             ).to_csv("project_predictions.csv", index=False)
train.assign(Cluster="Cluster " + pd.Series(kmeans.labels_, index=train.index).astype(str))[
    ["Student", "Cluster"] + features].to_csv("project_clusters.csv", index=False)
print("Saved project_predictions.csv and project_clusters.csv")

Saved project_predictions.csv and project_clusters.csv
